Transform Races Data
1. Read bronze races table
2. Keep only the columns requited for analytics (Drop wit, column)
3. Standardise column names using snake case race name, circuit)
4. Rename columns to make them more meaningful (date race date)
5. Remove duplicate records
6. Transform values of column race name to Title Case
7. Write the transformed data to silver races table

In [0]:
%run "/Workspace/Users/ganeshgadade157@gmail.com/Projects/Formula-f1/00.common/01.envioment_config"

In [0]:
from pyspark.sql import functions as f

folder_name = 'races'
file_name   = 'races.parquet'
silver_table = f"dev.silver.{folder_name}"

# 1. Read Bronze
raw_df = spark.read.format('parquet')\
              .load(f"{bronze_path}/{folder_name}/{file_name}")

# 2. Cast types + Drop url
raw_df = raw_df.withColumn('season', f.col('season').cast('integer'))\
               .withColumn('round', f.col('round').cast('integer'))\
               .withColumn('date',f.col('date').cast('date'))\
               .withColumn('ingestion_timestamp', f.col('ingestion_timestamp').cast('timestamp'))\
               .drop('url')                       

# 3. Rename columns
raw_df = raw_df.withColumnRenamed('raceName',   'race_name')\
               .withColumnRenamed('circuitId', 'circuit_id')

# 4. Filter null business key
raw_df = raw_df.filter(f.col('circuit_id').isNotNull())

# 5. Remove duplicates
raw_df = raw_df.dropDuplicates(['circuit_id'])     

# 6. Title Case
raw_df = raw_df.withColumn('race_name', f.initcap(f.col('race_name')))

# 7. Write to Silver
raw_df.write\
      .format('delta')\
      .mode('overwrite')\
      .saveAsTable(silver_table)

In [0]:
display(raw_df)